# O1-06 — High-Recall Detector Validation

## Objective

The Sep 25 milestone produced the first pre-inference Gamma compatibility
detector using a small high-confidence pilot dataset.

The objective of this stage is to validate and improve that detector using a
larger set of simulated and labelled cases.

The main priorities are:

1. expand the Gamma compatibility dataset;
2. test whether the previously observed failure patterns remain;
3. evaluate detector stability on more labelled examples;
4. tune the screening threshold for consistently high incompatible-case recall;
5. characterise remaining false negatives and difficult cases.

The Sep 25 Logistic Regression detector and threshold of 0.30 are treated as
provisional baselines rather than final production settings.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = Path(r"C:\AIML Y2 S2\Capstone 1\RYA")

FEATURE_FILE = BASE_DIR / "O1_classifier_inputs_unlabelled.csv"
OLD_LABEL_FILE = BASE_DIR / "O1_gamma_fit_pilot_zones_v1.csv"

print("Feature file exists:", FEATURE_FILE.exists())
print("Previous label file exists:", OLD_LABEL_FILE.exists())

Feature file exists: True
Previous label file exists: True


## 1. Load existing screening inputs and previous pilot labels

The full classifier-ready dataset contains the real instance inputs available
before neural-network inference.

The previous O1-04 pilot labels are loaded separately so that previously
simulated cases can be excluded from the new validation sample.

In [4]:
features = pd.read_csv(FEATURE_FILE)
old_labels = pd.read_csv(OLD_LABEL_FILE)

print("Full feature dataset:", features.shape)
print("Previous pilot labels:", old_labels.shape)

print("\nPrevious compatibility zones:")
print(
    old_labels["compatibility_zone_v1"]
    .value_counts()
)

Full feature dataset: (63328, 57)
Previous pilot labels: (175, 43)

Previous compatibility zones:
compatibility_zone_v1
compatible      89
ambiguous       51
incompatible    35
Name: count, dtype: int64


## 2. Inspect available cases for expanded validation

The 175 cases used in the Sep 25 pilot are excluded before selecting any new
simulation cases.

The remaining dataset is then characterised using the same risk dimensions
identified during the pilot:

- group-size imbalance;
- sparse response histograms;
- degenerate histograms;
- non-diagonal-mode unreliability matrices;
- regular cases.

This step is used only to design the expanded validation sample. It does not
assign Gamma-compatibility labels.

In [7]:
# Exclude all cases already used in the O1-04 pilot
old_exp_codes = set(old_labels["exp_code"])

available = features[
    ~features["exp_code"].isin(old_exp_codes)
].copy()

print("Full cases:", len(features))
print("Old pilot cases excluded:", len(features) - len(available))
print("Available new cases:", len(available))

Full cases: 63328
Old pilot cases excluded: 175
Available new cases: 63153


In [9]:
# Risk-region flags used only for sample design

available["flag_degenerate"] = (
    (available["degenerate_X"] == 1)
    | (available["degenerate_Y"] == 1)
)

available["flag_sparse"] = (
    (available["active_categories_X"] <= 2)
    | (available["active_categories_Y"] <= 2)
)

available["flag_ratio_10_50"] = (
    (available["group_size_ratio"] >= 10)
    & (available["group_size_ratio"] < 50)
)

available["flag_ratio_50_100"] = (
    (available["group_size_ratio"] >= 50)
    & (available["group_size_ratio"] < 100)
)

available["flag_ratio_100_plus"] = (
    available["group_size_ratio"] >= 100
)

available["flag_non_diagonal_P"] = (
    available["all_rows_diagonal_mode_P"] == 0
)

available["flag_regular"] = (
    (available["group_size_ratio"] < 10)
    & (~available["flag_sparse"])
    & (~available["flag_degenerate"])
    & (~available["flag_non_diagonal_P"])
)

In [11]:
risk_counts = pd.Series({
    "regular":
        available["flag_regular"].sum(),

    "ratio_10_50":
        available["flag_ratio_10_50"].sum(),

    "ratio_50_100":
        available["flag_ratio_50_100"].sum(),

    "ratio_100_plus":
        available["flag_ratio_100_plus"].sum(),

    "sparse":
        available["flag_sparse"].sum(),

    "degenerate":
        available["flag_degenerate"].sum(),

    "non_diagonal_P":
        available["flag_non_diagonal_P"].sum()
}).astype(int)

risk_counts

regular           42449
ratio_10_50       16554
ratio_50_100       2051
ratio_100_plus      589
sparse             1704
degenerate           42
non_diagonal_P     1158
dtype: int32

## 3. Build mutually exclusive validation strata

The risk flags overlap, so mutually exclusive strata are created for sample
selection.

A priority order is used so that rare and structurally important cases are
preserved first:

1. degenerate;
2. sparse but non-degenerate;
3. extreme imbalance (ratio >= 100);
4. high imbalance (50 <= ratio < 100);
5. moderate imbalance (10 <= ratio < 50);
6. non-diagonal-mode P matrices;
7. regular cases.

These strata are used only for balanced validation sampling and do not define
Gamma compatibility.

In [14]:
available["validation_stratum"] = "unassigned"

# 1. Degenerate — rarest group, highest priority
mask = available["flag_degenerate"]
available.loc[mask, "validation_stratum"] = "degenerate"

# 2. Sparse but not already degenerate
mask = (
    (available["validation_stratum"] == "unassigned")
    & available["flag_sparse"]
)
available.loc[mask, "validation_stratum"] = "sparse"

# 3. Extreme group-size imbalance
mask = (
    (available["validation_stratum"] == "unassigned")
    & available["flag_ratio_100_plus"]
)
available.loc[mask, "validation_stratum"] = "ratio_100_plus"

# 4. Ratio 50–100
mask = (
    (available["validation_stratum"] == "unassigned")
    & available["flag_ratio_50_100"]
)
available.loc[mask, "validation_stratum"] = "ratio_50_100"

# 5. Ratio 10–50
mask = (
    (available["validation_stratum"] == "unassigned")
    & available["flag_ratio_10_50"]
)
available.loc[mask, "validation_stratum"] = "ratio_10_50"

# 6. Non-diagonal-mode P not already captured above
mask = (
    (available["validation_stratum"] == "unassigned")
    & available["flag_non_diagonal_P"]
)
available.loc[mask, "validation_stratum"] = "non_diagonal_P"

# 7. Remaining regular / low-risk cases
mask = (
    available["validation_stratum"] == "unassigned"
)
available.loc[mask, "validation_stratum"] = "regular"

In [16]:
exclusive_counts = (
    available["validation_stratum"]
    .value_counts()
    .reindex([
        "degenerate",
        "sparse",
        "ratio_100_plus",
        "ratio_50_100",
        "ratio_10_50",
        "non_diagonal_P",
        "regular"
    ])
)

exclusive_counts

validation_stratum
degenerate           42
sparse             1662
ratio_100_plus      464
ratio_50_100       1759
ratio_10_50       15989
non_diagonal_P      788
regular           42449
Name: count, dtype: int64

## 4. Select the expanded 500-case validation sample

A new validation sample of 500 cases is selected from the unused instance pool.

The sampling design intentionally over-represents difficult input conditions
so that Gamma compatibility failures can be studied in sufficient numbers.

The sample is therefore a risk-enriched validation dataset and should not be
used to estimate the population prevalence of incompatible cases.

All 42 remaining degenerate cases are included because this condition is rare.
The other validation strata are randomly sampled using a fixed random seed for
reproducibility.

In [19]:
sample_targets = {
    "degenerate": 42,
    "sparse": 80,
    "ratio_100_plus": 90,
    "ratio_50_100": 80,
    "ratio_10_50": 70,
    "non_diagonal_P": 60,
    "regular": 78
}

print("Planned total:", sum(sample_targets.values()))

Planned total: 500


In [21]:
RANDOM_STATE = 42

sample_parts = []

for i, (stratum, n_target) in enumerate(
    sample_targets.items()
):
    stratum_pool = available[
        available["validation_stratum"] == stratum
    ].copy()

    if n_target > len(stratum_pool):
        raise ValueError(
            f"Requested {n_target} cases from {stratum}, "
            f"but only {len(stratum_pool)} are available."
        )

    # All cases are taken automatically when target == pool size.
    # Otherwise use reproducible random sampling.
    if n_target == len(stratum_pool):
        sampled = stratum_pool.copy()
    else:
        sampled = stratum_pool.sample(
            n=n_target,
            random_state=RANDOM_STATE + i
        )

    sample_parts.append(sampled)


validation_sample = (
    pd.concat(sample_parts, ignore_index=True)
    .sample(
        frac=1,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)

print("Validation sample shape:", validation_sample.shape)

print("\nSelected cases by stratum:")
print(
    validation_sample["validation_stratum"]
    .value_counts()
    .reindex(sample_targets.keys())
)

print(
    "\nUnique exp_codes:",
    validation_sample["exp_code"].nunique()
)

Validation sample shape: (500, 65)

Selected cases by stratum:
validation_stratum
degenerate        42
sparse            80
ratio_100_plus    90
ratio_50_100      80
ratio_10_50       70
non_diagonal_P    60
regular           78
Name: count, dtype: int64

Unique exp_codes: 500


In [23]:
# Check 1: no Sep 25 pilot cases accidentally reused
old_overlap = validation_sample[
    validation_sample["exp_code"].isin(
        old_labels["exp_code"]
    )
]

# Check 2: duplicate selected instances
duplicate_exp_codes = (
    validation_sample["exp_code"]
    .duplicated()
    .sum()
)

# Check 3: missing classifier inputs
missing_values = (
    validation_sample
    .isna()
    .sum()
    .sum()
)

print("Overlap with old 175-case pilot:", len(old_overlap))
print("Duplicate exp_codes:", duplicate_exp_codes)
print("Total missing values:", missing_values)

Overlap with old 175-case pilot: 0
Duplicate exp_codes: 0
Total missing values: 0


In [25]:
print("Cases by source file:")
print(
    validation_sample["source_file"]
    .value_counts()
)

print(
    "\nUnique P matrix questions:",
    validation_sample["P_matrix_question"]
    .nunique()
)

print("\nMost common P matrix questions:")
print(
    validation_sample["P_matrix_question"]
    .value_counts()
    .head(10)
)

Cases by source file:
source_file
orthog_B_4_feature_output_df.csv                  248
orthog_B_1_feature_output_df.csv                  214
orthog_B_3_large_classes_feature_output_df.csv     24
orthog_B_2_feature_output_df.csv                   14
Name: count, dtype: int64

Unique P matrix questions: 53

Most common P matrix questions:
P_matrix_question
cop1    67
ry30    17
sun4    14
ph3     13
sun1    13
ph2     13
phq5    13
ry10    12
ry21    12
ry18    11
Name: count, dtype: int64


In [27]:
VALIDATION_SAMPLE_FILE = (
    BASE_DIR / "O1_06_validation_sample_500.csv"
)

validation_sample.to_csv(
    VALIDATION_SAMPLE_FILE,
    index=False
)

print("Saved:")
print(VALIDATION_SAMPLE_FILE)

Saved:
C:\AIML Y2 S2\Capstone 1\RYA\O1_06_validation_sample_500.csv


### Validation sample QA conclusion

The expanded validation sample contains 500 new cases with no overlap with the
original 175-case pilot, no duplicate experiment IDs and no missing values.

The sample includes cases from all four source datasets and all 53 in-scope
unreliability-matrix questions.

The sample is intentionally risk-enriched rather than population-representative.
In particular, `cop1` is more heavily represented because difficult matrix and
input conditions were deliberately oversampled.

The selected 500 cases are now frozen for the Oct 9 validation experiment.

## 5. Reuse the O1-04 Gamma simulation engine

The same simulation and Gamma-fit implementation used for the Sep 25 pilot is
reused for the expanded validation dataset.

Keeping the statistical implementation unchanged allows the new 500-case
results to be compared directly with the previous 175-case pilot.

In [31]:
import sys
import inspect

if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

import gamma_fit_pilot

print("Helper file:")
print(gamma_fit_pilot.__file__)

print("\nrun_pilot signature:")
print(
    inspect.signature(
        gamma_fit_pilot.run_pilot
    )
)

Helper file:
C:\AIML Y2 S2\Capstone 1\RYA\gamma_fit_pilot.py

run_pilot signature:
(pilot_df: 'pd.DataFrame', num_trials: 'int' = 10000, base_seed: 'int' = 2026) -> 'pd.DataFrame'


In [32]:
[
    name
    for name in dir(gamma_fit_pilot)
    if not name.startswith("_")
]

['B_MATRIX',
 'annotations',
 'choose_pilot_instances',
 'fit_quality_metrics',
 'np',
 'pd',
 'proportions_to_counts',
 'row_to_p_matrix',
 'run_pilot',
 'sigma_matrices',
 'simulate_d_hat',
 'stats']

## 6. Smoke-test the expanded validation simulation

Before running all 500 validation cases with 10,000 trials each, a small
three-case smoke test is performed.

This confirms that the O1-04 simulation engine accepts the expanded validation
dataset and returns the expected Gamma-fit metrics without changing the
statistical implementation.

In [36]:
simulation_sample = validation_sample.copy()

# Keep compatibility with the O1-04 helper/output format
simulation_sample["pilot_group"] = (
    simulation_sample["validation_stratum"]
)

print("Simulation sample:", simulation_sample.shape)
print(
    simulation_sample[
        ["exp_code", "pilot_group"]
    ].head()
)

Simulation sample: (500, 66)
                       exp_code     pilot_group
0               orthog_B_4_1164     ratio_10_50
1               orthog_B_4_4962          sparse
2  orthog_B_3_large_classes_333  non_diagonal_P
3              orthog_B_4_27720  ratio_100_plus
4              orthog_B_4_28124          sparse


In [38]:
print("Smoke result shape:", smoke_results.shape)
print("\nColumns returned by run_pilot:")
print(smoke_results.columns.tolist())

NameError: name 'smoke_results' is not defined

In [ ]:
display(
    smoke_results[
        [
            "exp_code",
            "selection_group",
            "fit_failed",
            "valid_trials",
            "ks_stat",
            "q95_rel_error",
            "q99_rel_error",
            "cdf_error_rmse"
        ]
    ]
)

## 7. Run the expanded 10,000-trial Gamma validation

The smoke test completed successfully with no fitting failures.

The frozen 500-case validation sample is therefore evaluated using 10,000
Monte Carlo trials per case, matching the simulation scale used in the
original RYA workflow and the O1-04 pilot.

The experiment is executed in batches of 50 cases and each completed batch is
saved immediately. This allows the run to be resumed safely if execution is
interrupted.

In [42]:
import time

BATCH_SIZE = 50
NUM_TRIALS = 10_000
BASE_SEED = 2026

BATCH_DIR = BASE_DIR / "O1_06_batches"
BATCH_DIR.mkdir(exist_ok=True)

n_cases = len(simulation_sample)
n_batches = int(np.ceil(n_cases / BATCH_SIZE))

print("Total cases:", n_cases)
print("Trials per case:", NUM_TRIALS)
print("Batch size:", BATCH_SIZE)
print("Expected batches:", n_batches)
print("Batch directory:", BATCH_DIR)

Total cases: 500
Trials per case: 10000
Batch size: 50
Expected batches: 10
Batch directory: C:\AIML Y2 S2\Capstone 1\RYA\O1_06_batches


In [44]:
for batch_id in range(n_batches):

    start = batch_id * BATCH_SIZE
    end = min(
        start + BATCH_SIZE,
        n_cases
    )

    batch_number = batch_id + 1

    batch_file = (
        BATCH_DIR
        / f"O1_06_batch_{batch_number:02d}.csv"
    )

    # Safely resume if some batches already exist
    if batch_file.exists():
        print(
            f"Batch {batch_number}/{n_batches} "
            f"already exists — skipping."
        )
        continue

    batch_df = (
        simulation_sample
        .iloc[start:end]
        .copy()
    )

    print(
        f"\nRunning batch {batch_number}/{n_batches} "
        f"(cases {start + 1}–{end})"
    )

    start_time = time.time()

    batch_results = gamma_fit_pilot.run_pilot(
        batch_df,
        num_trials=NUM_TRIALS,
        base_seed=BASE_SEED + (batch_id * 10000)
    )

    batch_results.to_csv(
        batch_file,
        index=False
    )

    elapsed = time.time() - start_time

    print(
        f"Completed: {len(batch_results)} cases"
    )
    print(
        f"Fit failures: "
        f"{batch_results['fit_failed'].sum()}"
    )
    print(
        f"Saved: {batch_file}"
    )
    print(
        f"Batch time: {elapsed / 60:.2f} minutes"
    )

Batch 1/10 already exists — skipping.
Batch 2/10 already exists — skipping.
Batch 3/10 already exists — skipping.
Batch 4/10 already exists — skipping.
Batch 5/10 already exists — skipping.
Batch 6/10 already exists — skipping.
Batch 7/10 already exists — skipping.
Batch 8/10 already exists — skipping.
Batch 9/10 already exists — skipping.
Batch 10/10 already exists — skipping.


## 8. Combine and validate the 500-case Gamma results

All 500 validation cases completed successfully with no fitting failures.

The ten saved batch outputs are combined into one validation dataset.
The original validation strata are then reattached using `exp_code`.

Before assigning compatibility labels, the combined results are checked for
completeness, duplicate cases, missing fit-quality metrics and valid-trial
counts.

In [47]:
batch_files = sorted(
    BATCH_DIR.glob("O1_06_batch_*.csv")
)

print("Batch files found:", len(batch_files))

validation_results = pd.concat(
    [
        pd.read_csv(file)
        for file in batch_files
    ],
    ignore_index=True
)

print(
    "Combined result shape:",
    validation_results.shape
)

print(
    "Unique exp_codes:",
    validation_results["exp_code"].nunique()
)

print(
    "Total fit failures:",
    validation_results["fit_failed"].sum()
)

Batch files found: 10
Combined result shape: (500, 41)
Unique exp_codes: 500
Total fit failures: 0


In [49]:
validation_results = validation_results.drop(
    columns=["selection_group"],
    errors="ignore"
)

validation_results = validation_results.merge(
    validation_sample[
        [
            "exp_code",
            "validation_stratum"
        ]
    ],
    on="exp_code",
    how="left",
    validate="one_to_one"
)

print(
    validation_results[
        "validation_stratum"
    ].value_counts()
)

print(
    "\nMissing validation strata:",
    validation_results[
        "validation_stratum"
    ].isna().sum()
)

validation_stratum
ratio_100_plus    90
sparse            80
ratio_50_100      80
regular           78
ratio_10_50       70
non_diagonal_P    60
degenerate        42
Name: count, dtype: int64

Missing validation strata: 0


In [51]:
metric_cols = [
    "ks_stat",
    "q95_rel_error",
    "q99_rel_error",
    "tail_cdf_error_95",
    "tail_cdf_error_99",
    "cdf_error_rmse",
    "delta_bic_gamma_minus_weibull"
]

print("Missing fit-quality metrics:")
print(
    validation_results[
        metric_cols
    ].isna().sum()
)

print("\nValid trials summary:")
print(
    validation_results[
        "valid_trials"
    ].describe()
)

print(
    "\nCases with fewer than 10,000 valid trials:",
    (
        validation_results[
            "valid_trials"
        ] < 10_000
    ).sum()
)

Missing fit-quality metrics:
ks_stat                          0
q95_rel_error                    0
q99_rel_error                    0
tail_cdf_error_95                0
tail_cdf_error_99                0
cdf_error_rmse                   0
delta_bic_gamma_minus_weibull    0
dtype: int64

Valid trials summary:
count      500.000000
mean      9999.992000
std          0.141336
min       9997.000000
25%      10000.000000
50%      10000.000000
75%      10000.000000
max      10000.000000
Name: valid_trials, dtype: float64

Cases with fewer than 10,000 valid trials: 2


### QA conclusion

All 500 validation cases produced complete Gamma-fit quality metrics with no
fitting failures.

A total of 498 cases retained all 10,000 valid simulation trials. Two cases
retained slightly fewer trials, with the minimum being 9,997.

Because the reduction is negligible and all fit-quality metrics were computed
successfully, all 500 cases are retained for the Oct 9 validation analysis.

In [63]:
VALIDATION_RESULTS_FILE = (
    BASE_DIR
    / "O1_06_gamma_validation_500_metrics.csv"
)

validation_results.to_csv(
    VALIDATION_RESULTS_FILE,
    index=False
)

print("Saved:")
print(VALIDATION_RESULTS_FILE)

Saved:
C:\AIML Y2 S2\Capstone 1\RYA\O1_06_gamma_validation_500_metrics.csv


## 9. Compare the new 500-case validation results with the original pilot

The expanded validation dataset is compared with the original 175-case pilot
to test whether the same Gamma-fit failure patterns reproduce on new cases.

The comparison focuses on the main fit-quality measures used in O1-04:

- KS statistic;
- relative error at the 95th percentile;
- relative error at the 99th percentile;
- CDF RMSE.

Results are summarised by validation stratum before compatibility labels are
assigned.

In [55]:
new_group_summary = (
    validation_results
    .groupby("validation_stratum")
    .agg(
        n=("exp_code", "count"),

        ks_median=("ks_stat", "median"),
        ks_mean=("ks_stat", "mean"),
        ks_max=("ks_stat", "max"),

        q95_median=("q95_rel_error", "median"),
        q95_mean=("q95_rel_error", "mean"),
        q95_max=("q95_rel_error", "max"),

        q99_median=("q99_rel_error", "median"),
        q99_mean=("q99_rel_error", "mean"),
        q99_max=("q99_rel_error", "max"),

        cdf_rmse_median=("cdf_error_rmse", "median"),
        cdf_rmse_mean=("cdf_error_rmse", "mean")
    )
    .reset_index()
)

new_group_summary

,validation_stratum,n,ks_median,ks_mean,ks_max,q95_median,q95_mean,q95_max,q99_median,q99_mean,q99_max,cdf_rmse_median,cdf_rmse_mean
0,degenerate,42,0.066442,0.078846,0.264927,0.108711,0.111064,0.331037,0.219851,0.237004,0.477310,0.032419,0.037737
1,non_diagonal_P,60,0.016077,0.016155,0.030309,0.019781,0.023087,0.061834,0.054725,0.055855,0.143888,0.007180,0.007200
2,ratio_100_plus,90,0.082275,0.090338,0.221181,0.060531,0.084459,0.279844,0.137847,0.145938,0.488512,0.027728,0.030162
3,ratio_10_50,70,0.022290,0.026291,0.074629,0.035303,0.038530,0.148108,0.079515,0.084505,0.271990,0.010492,0.011772
4,ratio_50_100,80,0.055159,0.064086,0.183342,0.061728,0.067691,0.275235,0.127358,0.141183,0.509472,0.019115,0.024736
5,regular,78,0.018432,0.018783,0.038643,0.027127,0.026099,0.078335,0.049062,0.054818,0.191581,0.008123,0.008530
6,sparse,80,0.039573,0.057583,0.159347,0.060399,0.074783,0.293778,0.123050,0.146585,0.529301,0.020826,0.025716


### Replication finding

The expanded 500-case validation sample reproduced the main Gamma-fit patterns
observed in the original O1-04 pilot.

The highest Gamma-fit errors occurred in degenerate cases, followed by cases
with extreme group-size imbalance, sparse response histograms and group-size
ratios between 50 and 100.

Regular cases showed substantially lower fit error.

Cases selected because of non-diagonal-mode unreliability matrices also showed
low Gamma-fit error, suggesting that this matrix property alone is not a strong
indicator of Gamma incompatibility.

Overall, the new independent validation sample supports the earlier finding
that Gamma approximation quality is primarily associated with group-size and
response-distribution structure rather than non-diagonal-mode P matrices alone.

In [58]:
validation_results[
    [
        "ks_stat",
        "q95_rel_error",
        "q99_rel_error",
        "cdf_error_rmse"
    ]
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

,ks_stat,q95_rel_error,q99_rel_error,cdf_error_rmse
count,500.000000,500.000000,500.000000,500.000000
mean,0.050901,0.059564,0.119305,0.020514
std,0.043247,0.055230,0.097106,0.017246
min,0.004874,0.000088,0.000515,0.000727
50%,0.033252,0.042716,0.092354,0.014501
75%,0.069304,0.078388,0.161127,0.026982
90%,0.109966,0.137815,0.259498,0.042465
95%,0.144835,0.169894,0.307379,0.053412
99%,0.192156,0.256791,0.429601,0.083270
max,0.264927,0.331037,0.529301,0.111612


### Overall fit-quality distribution

Across the 500 new validation cases, the median 95th-percentile relative error
was approximately 4.3%.

The 90th percentile increased to approximately 13.8%, and the worst case
reached approximately 33.1%.

For the KS statistic, the median was approximately 0.033, while the 90th
percentile was approximately 0.110.

These distributions are broadly consistent with the thresholds explored in
O1-04. In particular, the previously used provisional boundaries around
0.05 for a cleaner-fit region and 0.10 for a poorer-fit region remain
reasonable for further validation, although they are still empirical rather
than universal cut-offs.

In [61]:
worst_new_q95 = (
    validation_results[
        [
            "exp_code",
            "validation_stratum",
            "n_X",
            "n_Y",
            "group_size_ratio",
            "active_categories_X",
            "active_categories_Y",
            "degenerate_X",
            "degenerate_Y",
            "P_matrix_question",
            "ks_stat",
            "q95_rel_error",
            "q99_rel_error",
            "cdf_error_rmse"
        ]
    ]
    .sort_values(
        "q95_rel_error",
        ascending=False
    )
    .head(15)
)

worst_new_q95

,exp_code,validation_stratum,n_X,n_Y,group_size_ratio,active_categories_X,active_categories_Y,degenerate_X,degenerate_Y,P_matrix_question,ks_stat,q95_rel_error,q99_rel_error,cdf_error_rmse
57,orthog_B_4_25255,degenerate,459,5,91.800000,4,1,0,1,ry27,0.231565,0.331037,0.411349,0.111612
317,orthog_B_1_18550,sparse,5,409,81.800000,2,4,0,0,phq7,0.159347,0.293778,0.367318,0.083236
224,orthog_B_4_752,ratio_100_plus,6,996,166.000000,3,4,0,0,sun3,0.221181,0.279844,0.488512,0.077823
272,orthog_B_4_19233,ratio_50_100,381,6,63.500000,4,3,0,0,phq5,0.182322,0.275235,0.487986,0.087670
247,orthog_B_1_23331,degenerate,5,179,35.800000,1,4,1,0,ry21,0.162301,0.259612,0.477310,0.095288
368,orthog_B_1_8766,ratio_100_plus,6,967,161.166667,3,4,0,0,ry18,0.175513,0.256763,0.144465,0.052328
301,orthog_B_4_5162,ratio_50_100,379,6,63.166667,4,3,0,0,phq3,0.127678,0.246683,0.509472,0.071267
443,orthog_B_1_22871,ratio_100_plus,5,851,170.200000,3,4,0,0,for1,0.128572,0.245352,0.230258,0.061662
80,orthog_B_1_3920,ratio_100_plus,899,5,179.800000,4,4,0,0,ry27,0.138897,0.243378,0.296688,0.052660
462,orthog_B_4_21966,sparse,210,8,26.250000,4,2,0,0,phq4,0.127018,0.227347,0.398307,0.068881


### Worst-case validation finding

The 15 largest 95th-percentile Gamma-fit errors were strongly concentrated in
the same difficult input regions identified in the original pilot.

Seven of the 15 worst cases had group-size ratios above 100, four were
degenerate cases, two were sparse cases and two had group-size ratios between
50 and 100.

No regular, moderate-imbalance or non-diagonal-P-only cases appeared among the
15 worst cases.

The worst cases also occurred across multiple unreliability matrices rather
than being concentrated in a single P matrix.

Together with the stratum-level results, this provides stronger evidence that
poor Gamma approximation is associated mainly with extreme group-size
imbalance and sparse or degenerate response distributions.

## 10. Apply the existing Gamma compatibility zones

The compatibility-zone definition developed in O1-04 is applied unchanged to
the 500 new validation cases.

The thresholds are not re-tuned using the new validation data.

The existing definition is:

- compatible: q95 relative error <= 0.05 AND KS <= 0.05;
- incompatible: q95 relative error > 0.10 OR KS > 0.10;
- ambiguous: all remaining cases.

Keeping the original thresholds unchanged provides an independent test of the
previous compatibility framework.

In [65]:
def assign_compatibility_zone(row):

    if (
        row["q95_rel_error"] <= 0.05
        and row["ks_stat"] <= 0.05
    ):
        return "compatible"

    elif (
        row["q95_rel_error"] > 0.10
        or row["ks_stat"] > 0.10
    ):
        return "incompatible"

    else:
        return "ambiguous"


validation_results["compatibility_zone_v1"] = (
    validation_results.apply(
        assign_compatibility_zone,
        axis=1
    )
)

print("Compatibility zones:")
print(
    validation_results[
        "compatibility_zone_v1"
    ].value_counts()
)

print("\nPercentages:")
print(
    validation_results[
        "compatibility_zone_v1"
    ]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Compatibility zones:
compatibility_zone_v1
compatible      244
ambiguous       147
incompatible    109
Name: count, dtype: int64

Percentages:
compatibility_zone_v1
compatible      48.8
ambiguous       29.4
incompatible    21.8
Name: proportion, dtype: float64


### Compatibility-zone replication

Applying the unchanged O1-04 compatibility thresholds to the 500 new validation
cases produced:

- 244 compatible cases (48.8%);
- 147 ambiguous cases (29.4%);
- 109 incompatible cases (21.8%).

This distribution is very similar to the original 175-case pilot, which contained
approximately 50.9% compatible, 29.1% ambiguous and 20.0% incompatible cases.

The similarity provides additional evidence that the provisional compatibility
framework is behaving consistently on previously unused RYA instance-data cases.

Because the validation sample is deliberately risk-enriched, these percentages
should not be interpreted as population prevalence across the full RYA dataset.

In [67]:
zone_by_stratum = pd.crosstab(
    validation_results["validation_stratum"],
    validation_results["compatibility_zone_v1"]
)

zone_by_stratum

compatibility_zone_v1,ambiguous,compatible,incompatible
validation_stratum,,,
degenerate,13,5,24
non_diagonal_P,3,57,0
ratio_100_plus,35,12,43
ratio_10_50,12,54,4
ratio_50_100,42,20,18
regular,6,72,0
sparse,36,24,20


### Compatibility patterns by validation stratum

The compatibility labels are strongly associated with the input-risk regions
identified earlier.

Degenerate cases showed the highest incompatible rate, with 24 of 42 cases
classified as incompatible. Extreme group-size imbalance also showed a high
incompatible rate, with 43 of 90 ratio-100-plus cases classified as
incompatible.

Sparse cases and cases with group-size ratios between 50 and 100 showed a
mixture of compatible, ambiguous and incompatible outcomes, indicating that
these regions may contain transitional or borderline Gamma behaviour.

In contrast, all 78 regular cases avoided the incompatible class, and none of
the 60 non-diagonal-P cases were classified as incompatible.

These results strengthen the evidence that Gamma incompatibility is associated
mainly with extreme group-size imbalance and sparse or degenerate response
distributions rather than non-diagonal-mode unreliability matrices alone.

In [69]:
zone_by_stratum_pct = (
    pd.crosstab(
        validation_results["validation_stratum"],
        validation_results["compatibility_zone_v1"],
        normalize="index"
    )
    .mul(100)
    .round(1)
)

zone_by_stratum_pct

compatibility_zone_v1,ambiguous,compatible,incompatible
validation_stratum,,,
degenerate,31.0,11.9,57.1
non_diagonal_P,5.0,95.0,0.0
ratio_100_plus,38.9,13.3,47.8
ratio_10_50,17.1,77.1,5.7
ratio_50_100,52.5,25.0,22.5
regular,7.7,92.3,0.0
sparse,45.0,30.0,25.0


In [71]:
VALIDATION_LABELLED_FILE = (
    BASE_DIR
    / "O1_06_gamma_validation_500_zones_v1.csv"
)

validation_results.to_csv(
    VALIDATION_LABELLED_FILE,
    index=False
)

print("Saved:")
print(VALIDATION_LABELLED_FILE)

Saved:
C:\AIML Y2 S2\Capstone 1\RYA\O1_06_gamma_validation_500_zones_v1.csv


### Section 10 conclusion

The expanded validation confirms that Gamma incompatibility is not distributed
uniformly across the RYA instance space.

The highest incompatible rates occur in degenerate cases and cases with extreme
group-size imbalance. Sparse cases and group-size ratios between 50 and 100
show mixed behaviour, with many cases falling into the ambiguous region.

Regular cases and non-diagonal-P-only cases show very low Gamma compatibility
risk in this validation sample.

This supports the use of pre-inference screening features based on group-size
imbalance and response-distribution structure.

The validation sample is risk-enriched, so these percentages describe the
stress-test sample and should not be interpreted as prevalence estimates for
the full RYA dataset.

## 11. Build the enlarged high-confidence detector dataset

The original O1-04 pilot and the new O1-06 validation set are combined to
create a larger labelled dataset for detector training.

Only high-confidence compatibility labels are used:

- compatible;
- incompatible.

Ambiguous cases are excluded from supervised training because their Gamma-fit
quality lies between the cleaner and poorer-fit regions.

The enlarged detector dataset uses the same pre-inference feature set as the
Sep 25 baseline so that model performance can be compared directly.

In [77]:
old_high_conf = (
    old_labels[
        old_labels["compatibility_zone_v1"]
        .isin(["compatible", "incompatible"])
    ][
        [
            "exp_code",
            "compatibility_zone_v1"
        ]
    ]
    .copy()
)

print("Old high-confidence cases:", old_high_conf.shape)

print(
    old_high_conf[
        "compatibility_zone_v1"
    ].value_counts()
)

Old high-confidence cases: (124, 2)
compatibility_zone_v1
compatible      89
incompatible    35
Name: count, dtype: int64


In [79]:
new_high_conf = (
    validation_results[
        validation_results["compatibility_zone_v1"]
        .isin(["compatible", "incompatible"])
    ][
        [
            "exp_code",
            "compatibility_zone_v1"
        ]
    ]
    .copy()
)

print("New high-confidence cases:", new_high_conf.shape)

print(
    new_high_conf[
        "compatibility_zone_v1"
    ].value_counts()
)

New high-confidence cases: (353, 2)
compatibility_zone_v1
compatible      244
incompatible    109
Name: count, dtype: int64


In [81]:
combined_labels = pd.concat(
    [
        old_high_conf,
        new_high_conf
    ],
    ignore_index=True
)

print(
    "Combined high-confidence labels:",
    combined_labels.shape
)

print(
    combined_labels[
        "compatibility_zone_v1"
    ].value_counts()
)

Combined high-confidence labels: (477, 2)
compatibility_zone_v1
compatible      333
incompatible    144
Name: count, dtype: int64


In [83]:
print(
    "Unique exp_codes:",
    combined_labels[
        "exp_code"
    ].nunique()
)

print(
    "Duplicate exp_codes:",
    combined_labels[
        "exp_code"
    ].duplicated().sum()
)

Unique exp_codes: 477
Duplicate exp_codes: 0


In [85]:
detector_expanded = combined_labels.merge(
    features,
    on="exp_code",
    how="left",
    validate="one_to_one"
)

print(
    "Expanded detector dataset:",
    detector_expanded.shape
)

print(
    "\nMissing values after merge:",
    detector_expanded.isna().sum().sum()
)

Expanded detector dataset: (477, 58)

Missing values after merge: 0


In [87]:
identifier_cols = [
    "exp_code",
    "P_matrix_question",
    "source_file"
]

target_col = "compatibility_zone_v1"

feature_cols = [
    col
    for col in detector_expanded.columns
    if col not in identifier_cols + [target_col]
]

X_expanded = (
    detector_expanded[
        feature_cols
    ]
    .copy()
)

y_expanded = (
    detector_expanded[
        target_col
    ]
    .map({
        "compatible": 0,
        "incompatible": 1
    })
)

print(
    "Expanded X shape:",
    X_expanded.shape
)

print(
    "Expanded y shape:",
    y_expanded.shape
)

print(
    "\nTarget counts:"
)

print(
    y_expanded.value_counts()
)

print(
    "\nNumber of features:",
    len(feature_cols)
)

print(
    "Missing values in X:",
    X_expanded.isna().sum().sum()
)

print(
    "Duplicate feature names:",
    X_expanded.columns.duplicated().sum()
)

print(
    "Non-numeric features:",
    X_expanded.select_dtypes(
        exclude=np.number
    ).columns.tolist()
)

Expanded X shape: (477, 54)
Expanded y shape: (477,)

Target counts:
compatibility_zone_v1
0    333
1    144
Name: count, dtype: int64

Number of features: 54
Missing values in X: 0
Duplicate feature names: 0
Non-numeric features: []


In [89]:
EXPANDED_DETECTOR_FILE = (
    BASE_DIR
    / "O1_06_detector_training_high_conf_477.csv"
)

detector_expanded.to_csv(
    EXPANDED_DETECTOR_FILE,
    index=False
)

print("Saved:")
print(EXPANDED_DETECTOR_FILE)

Saved:
C:\AIML Y2 S2\Capstone 1\RYA\O1_06_detector_training_high_conf_477.csv


## 12. Retrain baseline detectors on the expanded high-confidence dataset

The three baseline classifiers used in O1-05 are retrained on the enlarged
477-case high-confidence dataset.

The same 54 pre-inference features are used so that performance can be compared
directly with the Sep 25 baseline.

The primary evaluation objective remains recall of the incompatible class,
because false negatives would allow potentially unsafe cases to continue to
the existing Gamma neural-network pathway.

In [92]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models_expanded = {
    "Logistic Regression": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=5000,
                random_state=42
            )
        )
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

print("Models prepared:")
for name in models_expanded:
    print("-", name)

Models prepared:
- Logistic Regression
- Random Forest
- Gradient Boosting


In [94]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.metrics import (
    make_scorer,
    recall_score,
    precision_score,
    f1_score
)

recall_incompatible_scorer = make_scorer(
    recall_score,
    pos_label=1,
    zero_division=0
)

precision_incompatible_scorer = make_scorer(
    precision_score,
    pos_label=1,
    zero_division=0
)

f1_incompatible_scorer = make_scorer(
    f1_score,
    pos_label=1,
    zero_division=0
)

scoring = {
    "accuracy": "accuracy",
    "precision_incompatible":
        precision_incompatible_scorer,
    "recall_incompatible":
        recall_incompatible_scorer,
    "f1_incompatible":
        f1_incompatible_scorer
}

cv_expanded = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

cv_expanded_rows = []

for name, model in models_expanded.items():

    scores = cross_validate(
        model,
        X_expanded,
        y_expanded,
        cv=cv_expanded,
        scoring=scoring,
        n_jobs=-1
    )

    cv_expanded_rows.append({
        "model": name,

        "accuracy_mean":
            scores["test_accuracy"].mean(),

        "accuracy_std":
            scores["test_accuracy"].std(),

        "precision_incompatible_mean":
            scores[
                "test_precision_incompatible"
            ].mean(),

        "precision_incompatible_std":
            scores[
                "test_precision_incompatible"
            ].std(),

        "recall_incompatible_mean":
            scores[
                "test_recall_incompatible"
            ].mean(),

        "recall_incompatible_std":
            scores[
                "test_recall_incompatible"
            ].std(),

        "f1_incompatible_mean":
            scores[
                "test_f1_incompatible"
            ].mean(),

        "f1_incompatible_std":
            scores[
                "test_f1_incompatible"
            ].std()
    })

cv_expanded_summary = pd.DataFrame(
    cv_expanded_rows
)

cv_expanded_summary.sort_values(
    "recall_incompatible_mean",
    ascending=False
)

,model,accuracy_mean,accuracy_std,precision_incompatible_mean,precision_incompatible_std,recall_incompatible_mean,recall_incompatible_std,f1_incompatible_mean,f1_incompatible_std
0,Logistic Regression,0.915300,0.024068,0.832104,0.055171,0.907512,0.055343,0.866196,0.037535
1,Random Forest,0.905022,0.021261,0.852180,0.047550,0.835148,0.083136,0.839930,0.041931
2,Gradient Boosting,0.898116,0.019178,0.837017,0.048177,0.829039,0.067577,0.830164,0.034037


### Expanded cross-validation finding

Logistic Regression remained the strongest baseline for detecting Gamma-
incompatible cases after expanding the high-confidence labelled dataset from
124 to 477 cases.

Its mean incompatible-case recall increased from approximately 0.851 in the
Sep 25 pilot to 0.908 in the expanded evaluation.

The recall standard deviation also decreased substantially, indicating more
stable performance across repeated validation splits.

Random Forest and Gradient Boosting also improved compared with the original
pilot, but both continued to show lower incompatible-case recall than Logistic
Regression.

Therefore, Logistic Regression remains the primary candidate for the
high-recall pre-inference screening detector.

## 13. External validation of the Sep 25 detector

Before re-tuning the detector using the enlarged dataset, the Sep 25 Logistic
Regression detector configuration is evaluated on the newly generated
high-confidence validation cases.

The original 124 high-confidence pilot cases are used for model fitting, while
the 353 high-confidence cases obtained from the independent 500-case validation
experiment are used only for testing.

The original provisional decision threshold of 0.30 is retained unchanged.

This provides a direct test of whether the Sep 25 detector and operating
threshold generalise to previously unseen RYA instance-data cases.

In [98]:
old_detector_df = old_high_conf.merge(
    features,
    on="exp_code",
    how="left",
    validate="one_to_one"
)

new_detector_df = new_high_conf.merge(
    features,
    on="exp_code",
    how="left",
    validate="one_to_one"
)

X_old = old_detector_df[feature_cols].copy()

y_old = (
    old_detector_df["compatibility_zone_v1"]
    .map({
        "compatible": 0,
        "incompatible": 1
    })
)

X_new = new_detector_df[feature_cols].copy()

y_new = (
    new_detector_df["compatibility_zone_v1"]
    .map({
        "compatible": 0,
        "incompatible": 1
    })
)

print("Sep 25 training cases:", X_old.shape)
print(y_old.value_counts())

print("\nNew external validation cases:", X_new.shape)
print(y_new.value_counts())

Sep 25 training cases: (124, 54)
compatibility_zone_v1
0    89
1    35
Name: count, dtype: int64

New external validation cases: (353, 54)
compatibility_zone_v1
0    244
1    109
Name: count, dtype: int64


In [100]:
from sklearn.base import clone
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

sep25_model = clone(
    models_expanded["Logistic Regression"]
)

sep25_model.fit(
    X_old,
    y_old
)

new_probabilities = sep25_model.predict_proba(
    X_new
)[:, 1]

SEP25_THRESHOLD = 0.30

new_predictions = (
    new_probabilities >= SEP25_THRESHOLD
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_new,
    new_predictions,
    labels=[0, 1]
).ravel()

print("Sep 25 threshold:", SEP25_THRESHOLD)

print("\nExternal validation confusion matrix:")
print(
    confusion_matrix(
        y_new,
        new_predictions,
        labels=[0, 1]
    )
)

print("\nExternal validation metrics:")
print(
    "Accuracy:",
    round(
        accuracy_score(
            y_new,
            new_predictions
        ),
        4
    )
)

print(
    "Incompatible precision:",
    round(
        precision_score(
            y_new,
            new_predictions,
            pos_label=1,
            zero_division=0
        ),
        4
    )
)

print(
    "Incompatible recall:",
    round(
        recall_score(
            y_new,
            new_predictions,
            pos_label=1,
            zero_division=0
        ),
        4
    )
)

print(
    "Incompatible F1:",
    round(
        f1_score(
            y_new,
            new_predictions,
            pos_label=1,
            zero_division=0
        ),
        4
    )
)

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

Sep 25 threshold: 0.3

External validation confusion matrix:
[[221  23]
 [ 15  94]]

External validation metrics:
Accuracy: 0.8924
Incompatible precision: 0.8034
Incompatible recall: 0.8624
Incompatible F1: 0.8319

TN: 221
FP: 23
FN: 15
TP: 94


### External validation finding

The Sep 25 Logistic Regression detector was evaluated without changing its
original probability threshold of 0.30.

On 353 previously unseen high-confidence validation cases, the detector
achieved:

- accuracy = 0.892;
- incompatible precision = 0.803;
- incompatible recall = 0.862;
- incompatible F1 = 0.832;
- false negatives = 15;
- false positives = 23.

The detector therefore generalised reasonably well to new RYA instance-data
cases, but the original 0.30 threshold did not maintain the zero-false-negative
result observed in the smaller Sep 25 pilot.

This demonstrates why the larger validation experiment was necessary. The
Sep 25 result should be interpreted as an encouraging pilot result rather than
evidence of guaranteed 100% recall.

The 15 newly identified false negatives provide important evidence for the
Oct 9 objective of improving high-recall detection and characterising the
remaining difficult cases.

## 14. Characterise false negatives from external validation

False negatives are the most important detector errors because they represent
Gamma-incompatible cases that would incorrectly pass through the screening gate
to the existing neural-network pathway.

The 15 false negatives from the independent validation experiment are therefore
examined in detail before any detector re-tuning is performed.

In [104]:
external_eval = new_detector_df[
    [
        "exp_code",
        "P_matrix_question",
        "source_file",
        "group_size_ratio",
        "active_categories_X",
        "active_categories_Y",
        "degenerate_X",
        "degenerate_Y",
        "sparse_count_X",
        "sparse_count_Y",
        "asymmetry_P",
        "all_rows_diagonal_mode_P"
    ]
].copy()

external_eval["true_label"] = y_new.to_numpy()

external_eval["prob_incompatible"] = (
    new_probabilities
)

external_eval["predicted_label"] = (
    new_predictions
)

false_negatives_external = external_eval[
    (external_eval["true_label"] == 1)
    & (external_eval["predicted_label"] == 0)
].copy()

print(
    "External-validation false negatives:",
    len(false_negatives_external)
)

display(
    false_negatives_external.sort_values(
        "prob_incompatible",
        ascending=False
    )
)

External-validation false negatives: 15


,exp_code,P_matrix_question,source_file,group_size_ratio,active_categories_X,active_categories_Y,degenerate_X,degenerate_Y,sparse_count_X,sparse_count_Y,asymmetry_P,all_rows_diagonal_mode_P,true_label,prob_incompatible,predicted_label
259,orthog_B_4_23045,ry8,orthog_B_4_feature_output_df.csv,60.285714,4,3,0,0,0,1,0.018480,1,1,0.252178,0
269,orthog_B_4_19138,ry24,orthog_B_4_feature_output_df.csv,55.600000,4,1,0,1,0,3,0.017465,1,1,0.251185,0
88,orthog_B_1_12706,tru1,orthog_B_1_feature_output_df.csv,54.384615,3,4,0,0,1,0,0.010457,1,1,0.211637,0
84,orthog_B_4_2781,ry30,orthog_B_4_feature_output_df.csv,119.714286,4,4,0,0,0,0,0.006258,1,1,0.171771,0
328,orthog_B_4_21966,phq4,orthog_B_4_feature_output_df.csv,26.250000,4,2,0,0,0,2,0.005955,1,1,0.103505,0
170,orthog_B_3_large_classes_1675,ry5,orthog_B_3_large_classes_feature_output_df.csv,144.000000,4,4,0,0,0,0,0.010361,1,1,0.068835,0
37,orthog_B_4_13896,ry10,orthog_B_4_feature_output_df.csv,111.000000,4,3,0,0,0,1,0.018557,1,1,0.046860,0
53,orthog_B_1_4007,ry30,orthog_B_1_feature_output_df.csv,33.363636,4,3,0,0,0,1,0.005900,1,1,0.039301,0
182,orthog_B_1_3383,for2,orthog_B_1_feature_output_df.csv,110.333333,3,4,0,0,1,0,0.013741,1,1,0.030505,0
173,orthog_B_4_24919,ry22,orthog_B_4_feature_output_df.csv,38.400000,4,2,0,0,1,2,0.006854,1,1,0.024397,0


### False-negative characterisation

The 15 false negatives show that the Sep 25 detector does not fail only at
cases immediately below the 0.30 probability threshold.

Several missed incompatible cases received probabilities close to the threshold,
suggesting that threshold adjustment may recover them.

However, other incompatible cases received very low predicted probabilities,
including values below 0.05. Therefore, threshold adjustment alone is unlikely
to eliminate all false negatives.

The missed cases include a mixture of extreme group-size imbalance, moderate-to-
high imbalance, sparse response distributions and some fully active histograms.

This indicates that Gamma incompatibility is driven by combinations of input
conditions rather than by a single simple rule.

The expanded labelled dataset should therefore be used both to retrain the
decision boundary and to tune the operating threshold.

In [106]:
false_negatives_external = (
    false_negatives_external
    .merge(
        validation_results[
            [
                "exp_code",
                "validation_stratum",
                "ks_stat",
                "q95_rel_error",
                "q99_rel_error",
                "cdf_error_rmse"
            ]
        ],
        on="exp_code",
        how="left",
        validate="one_to_one"
    )
)

print("False negatives by validation stratum:")
print(
    false_negatives_external[
        "validation_stratum"
    ].value_counts()
)

display(
    false_negatives_external[
        [
            "exp_code",
            "validation_stratum",
            "prob_incompatible",
            "group_size_ratio",
            "active_categories_X",
            "active_categories_Y",
            "degenerate_X",
            "degenerate_Y",
            "sparse_count_X",
            "sparse_count_Y",
            "P_matrix_question",
            "ks_stat",
            "q95_rel_error",
            "q99_rel_error"
        ]
    ].sort_values(
        "prob_incompatible",
        ascending=False
    )
)

False negatives by validation stratum:
validation_stratum
ratio_50_100      6
ratio_100_plus    5
sparse            2
ratio_10_50       1
degenerate        1
Name: count, dtype: int64


,exp_code,validation_stratum,prob_incompatible,group_size_ratio,active_categories_X,active_categories_Y,degenerate_X,degenerate_Y,sparse_count_X,sparse_count_Y,P_matrix_question,ks_stat,q95_rel_error,q99_rel_error
11,orthog_B_4_23045,ratio_50_100,0.252178,60.285714,4,3,0,0,0,1,ry8,0.101304,0.045548,0.134767
12,orthog_B_4_19138,degenerate,0.251185,55.600000,4,1,0,1,0,3,ry24,0.066541,0.136181,0.243475
4,orthog_B_1_12706,ratio_50_100,0.211637,54.384615,3,4,0,0,1,0,tru1,0.124381,0.149387,0.217248
3,orthog_B_4_2781,ratio_100_plus,0.171771,119.714286,4,4,0,0,0,0,ry30,0.054586,0.131045,0.132688
14,orthog_B_4_21966,sparse,0.103505,26.250000,4,2,0,0,0,2,phq4,0.127018,0.227347,0.398307
6,orthog_B_3_large_classes_1675,ratio_100_plus,0.068835,144.000000,4,4,0,0,0,0,ry5,0.068638,0.172623,0.125892
0,orthog_B_4_13896,ratio_100_plus,0.046860,111.000000,4,3,0,0,0,1,ry10,0.057270,0.118559,0.049068
1,orthog_B_1_4007,ratio_10_50,0.039301,33.363636,4,3,0,0,0,1,ry30,0.057750,0.117002,0.191488
8,orthog_B_1_3383,ratio_100_plus,0.030505,110.333333,3,4,0,0,1,0,for2,0.079693,0.115659,0.114889
7,orthog_B_4_24919,sparse,0.024397,38.400000,4,2,0,0,1,2,ry22,0.148951,0.153699,0.215614


### Section 14 conclusion

False negatives were concentrated mainly in moderate-to-extreme group-size
imbalance.

Of the 15 incompatible cases missed by the Sep 25 detector:

- 6 were in the ratio 50–100 stratum;
- 5 were in the ratio >=100 stratum;
- 2 were sparse;
- 1 was in the ratio 10–50 stratum;
- 1 was degenerate.

Therefore, 11 of the 15 false negatives occurred at group-size ratios of at
least 50.

Some missed cases had sparse response distributions, but several retained
three or four active categories and were not degenerate. This confirms that
Gamma incompatibility cannot be identified reliably using a single simple
input rule.

The results support retraining the classifier using the enlarged labelled
dataset rather than relying only on a lower probability threshold.

## 15. Tune the expanded detector for high incompatible-case recall

The external validation result for the Sep 25 detector is retained as a fixed
historical evaluation.

The original and new high-confidence cases are now combined into the enlarged
477-case development dataset.

Repeated out-of-fold probabilities are generated for Logistic Regression so
that a revised operating threshold can be selected for high incompatible-case
recall without evaluating cases using models that were trained on those same
cases.

The objective is to substantially reduce false negatives while monitoring the
increase in false positives.

In [111]:
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold

cv_threshold_expanded = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

expanded_probability_records = []

for fold_id, (train_idx, test_idx) in enumerate(
    cv_threshold_expanded.split(
        X_expanded,
        y_expanded
    ),
    start=1
):

    model = clone(
        models_expanded["Logistic Regression"]
    )

    model.fit(
        X_expanded.iloc[train_idx],
        y_expanded.iloc[train_idx]
    )

    probs = model.predict_proba(
        X_expanded.iloc[test_idx]
    )[:, 1]

    for idx, prob in zip(
        test_idx,
        probs
    ):
        expanded_probability_records.append({
            "row_index": idx,
            "true_label": int(
                y_expanded.iloc[idx]
            ),
            "prob_incompatible": float(prob),
            "fold_id": fold_id
        })


expanded_oof = pd.DataFrame(
    expanded_probability_records
)

print(
    "Prediction records:",
    expanded_oof.shape
)

print("\nPredictions per case:")
print(
    expanded_oof[
        "row_index"
    ]
    .value_counts()
    .describe()
)

Prediction records: (4770, 4)

Predictions per case:
count    477.0
mean      10.0
std        0.0
min       10.0
25%       10.0
50%       10.0
75%       10.0
max       10.0
Name: count, dtype: float64


In [113]:
expanded_mean_oof = (
    expanded_oof
    .groupby(
        [
            "row_index",
            "true_label"
        ],
        as_index=False
    )
    .agg(
        mean_prob_incompatible=(
            "prob_incompatible",
            "mean"
        ),
        std_prob_incompatible=(
            "prob_incompatible",
            "std"
        )
    )
)

print(
    "Averaged cases:",
    expanded_mean_oof.shape
)

print("\nProbability by true class:")
print(
    expanded_mean_oof.groupby(
        "true_label"
    )[
        "mean_prob_incompatible"
    ].describe()
)

Averaged cases: (477, 4)

Probability by true class:
            count      mean       std           min       25%       50%  \
true_label                                                                
0           333.0  0.108985  0.224074  5.867930e-14  0.000121  0.003571   
1           144.0  0.861795  0.207734  6.924903e-02  0.832815  0.958324   

                 75%       max  
true_label                      
0           0.079702  0.977826  
1           0.989701  1.000000  


In [115]:
thresholds_expanded = [
    0.50,
    0.45,
    0.40,
    0.35,
    0.30,
    0.25,
    0.20,
    0.15,
    0.10,
    0.05
]

expanded_threshold_rows = []

for threshold in thresholds_expanded:

    y_true = (
        expanded_mean_oof[
            "true_label"
        ]
    )

    y_pred = (
        expanded_mean_oof[
            "mean_prob_incompatible"
        ] >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    expanded_threshold_rows.append({
        "threshold": threshold,

        "recall_incompatible":
            recall_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            ),

        "precision_incompatible":
            precision_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            ),

        "f1_incompatible":
            f1_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            ),

        "false_negatives": fn,
        "false_positives": fp,
        "true_positives": tp,
        "true_negatives": tn
    })


expanded_threshold_df = pd.DataFrame(
    expanded_threshold_rows
)

expanded_threshold_df

,threshold,recall_incompatible,precision_incompatible,f1_incompatible,false_negatives,false_positives,true_positives,true_negatives
0,0.50,0.916667,0.830189,0.871287,12,27,132,306
1,0.45,0.937500,0.823171,0.876623,9,29,135,304
2,0.40,0.944444,0.804734,0.869010,8,33,136,300
3,0.35,0.958333,0.775281,0.857143,6,40,138,293
4,0.30,0.965278,0.743316,0.839879,5,48,139,285
5,0.25,0.965278,0.723958,0.827381,5,53,139,280
6,0.20,0.972222,0.707071,0.818713,4,58,140,275
7,0.15,0.979167,0.674641,0.798867,3,68,141,265
8,0.10,0.993056,0.647059,0.783562,1,78,143,255
9,0.05,1.000000,0.605042,0.753927,0,94,144,239


### High-recall threshold selection

The expanded Logistic Regression detector was evaluated across a range of
probability thresholds using repeated out-of-fold predictions.

At the original Sep 25 threshold of 0.30, the expanded detector achieved
approximately 96.5% incompatible-case recall, with five false negatives.

Lowering the threshold to 0.10 increased incompatible-case recall to
approximately 99.3%, leaving one false negative among 144 incompatible cases.

A further reduction to 0.05 eliminated the final false negative but increased
false positives from 78 to 94.

Therefore, 0.10 is selected as the provisional Oct 9 high-recall operating
threshold because it provides near-complete detection of incompatible cases
while avoiding the larger false-positive increase required for a 0.05
threshold.

This threshold remains provisional and should be validated further before
production deployment.

In [118]:
OCT9_THRESHOLD = 0.10

print(
    "Provisional Oct 9 high-recall threshold:",
    OCT9_THRESHOLD
)

Provisional Oct 9 high-recall threshold: 0.1


## 16. Inspect the remaining false negative at the Oct 9 threshold

At the provisional high-recall threshold of 0.10, the expanded Logistic
Regression detector correctly identifies 143 of 144 incompatible cases under
repeated out-of-fold evaluation.

The single remaining false negative is inspected in detail because it
represents the most important unresolved detector failure in the current
development dataset.

In [121]:
expanded_mean_oof_with_info = (
    expanded_mean_oof
    .merge(
        detector_expanded[
            [
                "exp_code",
                "P_matrix_question",
                "source_file",
                "group_size_ratio",
                "active_categories_X",
                "active_categories_Y",
                "degenerate_X",
                "degenerate_Y",
                "sparse_count_X",
                "sparse_count_Y",
                "asymmetry_P",
                "all_rows_diagonal_mode_P",
                "compatibility_zone_v1"
            ]
        ].reset_index().rename(
            columns={"index": "row_index"}
        ),
        on="row_index",
        how="left",
        validate="one_to_one"
    )
)

expanded_mean_oof_with_info["pred_oct9"] = (
    expanded_mean_oof_with_info[
        "mean_prob_incompatible"
    ] >= OCT9_THRESHOLD
).astype(int)

remaining_fn = expanded_mean_oof_with_info[
    (expanded_mean_oof_with_info["true_label"] == 1)
    & (expanded_mean_oof_with_info["pred_oct9"] == 0)
].copy()

print(
    "Remaining false negatives:",
    len(remaining_fn)
)

display(
    remaining_fn[
        [
            "exp_code",
            "mean_prob_incompatible",
            "std_prob_incompatible",
            "group_size_ratio",
            "active_categories_X",
            "active_categories_Y",
            "degenerate_X",
            "degenerate_Y",
            "sparse_count_X",
            "sparse_count_Y",
            "P_matrix_question",
            "asymmetry_P",
            "all_rows_diagonal_mode_P"
        ]
    ]
)

Remaining false negatives: 1


,exp_code,mean_prob_incompatible,std_prob_incompatible,group_size_ratio,active_categories_X,active_categories_Y,degenerate_X,degenerate_Y,sparse_count_X,sparse_count_Y,P_matrix_question,asymmetry_P,all_rows_diagonal_mode_P
284,orthog_B_4_5543,0.069249,0.039967,66.75,4,4,0,0,0,0,ry11,0.011344,1


In [123]:
remaining_fn_detail = remaining_fn.merge(
    pd.concat([
        old_labels[
            [
                "exp_code",
                "ks_stat",
                "q95_rel_error",
                "q99_rel_error",
                "cdf_error_rmse"
            ]
        ],
        validation_results[
            [
                "exp_code",
                "ks_stat",
                "q95_rel_error",
                "q99_rel_error",
                "cdf_error_rmse"
            ]
        ]
    ]).drop_duplicates(
        subset="exp_code"
    ),
    on="exp_code",
    how="left",
    validate="one_to_one"
)

display(
    remaining_fn_detail[
        [
            "exp_code",
            "mean_prob_incompatible",
            "std_prob_incompatible",
            "group_size_ratio",
            "active_categories_X",
            "active_categories_Y",
            "degenerate_X",
            "degenerate_Y",
            "P_matrix_question",
            "ks_stat",
            "q95_rel_error",
            "q99_rel_error",
            "cdf_error_rmse"
        ]
    ]
)

,exp_code,mean_prob_incompatible,std_prob_incompatible,group_size_ratio,active_categories_X,active_categories_Y,degenerate_X,degenerate_Y,P_matrix_question,ks_stat,q95_rel_error,q99_rel_error,cdf_error_rmse
0,orthog_B_4_5543,0.069249,0.039967,66.75,4,4,0,0,ry11,0.025782,0.120671,0.018999,0.012601


### Remaining false-negative interpretation

The single remaining false negative is `orthog_B_4_5543`.

This case does not show the obvious structural risk characteristics seen in
many other incompatible cases. Both response histograms use all four
categories, neither histogram is degenerate or sparse, and the unreliability
matrix remains diagonal-mode.

The main visible risk factor is a group-size ratio of 66.75.

Its Gamma-fit evidence is also mixed. The KS statistic (0.026), q99 relative
error (0.019), and CDF RMSE (0.013) are relatively low, while the q95 relative
error (0.121) exceeds the provisional incompatibility threshold of 0.10.

Therefore, this case appears closer to the compatibility boundary than many
other incompatible examples.

Before changing the detector threshold or feature set to capture this single
case, the stability of its Gamma-compatibility label should be checked across
multiple simulation seeds.

## 17. Stability check for the remaining false-negative label

The remaining false negative lies relatively close to the empirical
compatibility boundary.

To test whether its incompatible label is stable rather than a consequence of
Monte Carlo variation, the same instance is simulated repeatedly using
different random seeds while keeping 10,000 trials per run.

No input features or compatibility thresholds are changed.

In [127]:
remaining_case = simulation_sample[
    simulation_sample["exp_code"] == "orthog_B_4_5543"
].copy()

print("Cases found:", len(remaining_case))

display(
    remaining_case[
        [
            "exp_code",
            "validation_stratum",
            "group_size_ratio",
            "active_categories_X",
            "active_categories_Y",
            "degenerate_X",
            "degenerate_Y"
        ]
    ]
)

Cases found: 1


,exp_code,validation_stratum,group_size_ratio,active_categories_X,active_categories_Y,degenerate_X,degenerate_Y
226,orthog_B_4_5543,ratio_50_100,66.75,4,4,0,0


In [129]:
stability_results = []

stability_seeds = [
    3001, 3002, 3003, 3004, 3005,
    3006, 3007, 3008, 3009, 3010
]

for seed in stability_seeds:

    result = gamma_fit_pilot.run_pilot(
        remaining_case,
        num_trials=10_000,
        base_seed=seed
    )

    row = result.iloc[0]

    stability_results.append({
        "seed": seed,
        "valid_trials": row["valid_trials"],
        "ks_stat": row["ks_stat"],
        "q95_rel_error": row["q95_rel_error"],
        "q99_rel_error": row["q99_rel_error"],
        "cdf_error_rmse": row["cdf_error_rmse"]
    })

stability_df = pd.DataFrame(
    stability_results
)

stability_df

,seed,valid_trials,ks_stat,q95_rel_error,q99_rel_error,cdf_error_rmse
0,3001,10000,0.021781,0.110216,0.007699,0.011421
1,3002,10000,0.023945,0.129376,0.014987,0.012166
2,3003,10000,0.026861,0.127264,0.017040,0.012569
3,3004,10000,0.024405,0.114192,0.010285,0.011874
4,3005,10000,0.026376,0.122142,0.006367,0.013061
5,3006,10000,0.022133,0.108775,0.007991,0.011298
6,3007,10000,0.024685,0.120670,0.009473,0.013153
7,3008,10000,0.023443,0.107391,0.000056,0.011656
8,3009,10000,0.023351,0.121784,0.017692,0.012055
9,3010,10000,0.022578,0.102067,0.008692,0.011257


In [131]:
def stability_zone(row):

    if (
        row["q95_rel_error"] <= 0.05
        and row["ks_stat"] <= 0.05
    ):
        return "compatible"

    elif (
        row["q95_rel_error"] > 0.10
        or row["ks_stat"] > 0.10
    ):
        return "incompatible"

    else:
        return "ambiguous"


stability_df["compatibility_zone_v1"] = (
    stability_df.apply(
        stability_zone,
        axis=1
    )
)

print("Zone counts across repeated simulations:")
print(
    stability_df[
        "compatibility_zone_v1"
    ].value_counts()
)

print("\nMetric summary:")
display(
    stability_df[
        [
            "ks_stat",
            "q95_rel_error",
            "q99_rel_error",
            "cdf_error_rmse"
        ]
    ].describe()
)

Zone counts across repeated simulations:
compatibility_zone_v1
incompatible    10
Name: count, dtype: int64

Metric summary:


,ks_stat,q95_rel_error,q99_rel_error,cdf_error_rmse
count,10.000000,10.000000,10.000000,10.000000
mean,0.023956,0.116388,0.010028,0.012051
std,0.001687,0.009156,0.005344,0.000691
min,0.021781,0.102067,0.000056,0.011257
25%,0.022772,0.109135,0.007772,0.011480
50%,0.023694,0.117431,0.009083,0.011964
75%,0.024615,0.122053,0.013811,0.012468
max,0.026861,0.129376,0.017692,0.013153


### Stability-check conclusion

The remaining false-negative case was re-simulated ten times using independent
random seeds and 10,000 trials per run.

In all ten runs, the q95 relative error remained above the provisional 0.10
incompatibility boundary, ranging from approximately 0.102 to 0.129.

Therefore, the incompatible label is stable and is not explained by Monte Carlo
variation.

The case is unusual because it does not contain sparse or degenerate response
histograms and its unreliability matrix does not show an obvious structural
warning. Its main visible risk factor is a group-size ratio of 66.75.

This represents a genuine remaining limitation of the current detector rather
than an unstable simulation label.

## 18. Save Oct 9 high-recall detector results

The expanded validation and detector-tuning stage is now complete.

Key outcomes:

- 500 new RYA instance-data cases were simulated using 10,000 trials each;
- the original Gamma-fit failure patterns reproduced on the larger validation set;
- 353 new high-confidence labels were obtained;
- the detector development dataset increased from 124 to 477 high-confidence cases;
- Logistic Regression remained the strongest recall-oriented baseline;
- the Sep 25 detector achieved 86.2% incompatible recall on the new external
  validation cases;
- retraining on the expanded labelled dataset increased detector stability;
- a provisional threshold of 0.10 achieved 99.3% incompatible recall under
  repeated out-of-fold evaluation;
- 143 of 144 incompatible cases were detected;
- the remaining false negative was shown to be a stable difficult case rather
  than a Monte Carlo labelling artefact.

The 0.10 threshold remains provisional and should be treated as the current
high-recall development setting rather than a final production threshold.

In [135]:
OCT9_CV_FILE = (
    BASE_DIR
    / "O1_06_expanded_cross_validation_summary.csv"
)

OCT9_THRESHOLD_FILE = (
    BASE_DIR
    / "O1_06_expanded_threshold_analysis.csv"
)

OCT9_FN_FILE = (
    BASE_DIR
    / "O1_06_remaining_false_negative.csv"
)

OCT9_STABILITY_FILE = (
    BASE_DIR
    / "O1_06_remaining_false_negative_stability.csv"
)

EXPANDED_DETECTOR_FILE = (
    BASE_DIR
    / "O1_06_detector_training_high_conf_477.csv"
)

cv_expanded_summary.to_csv(
    OCT9_CV_FILE,
    index=False
)

expanded_threshold_df.to_csv(
    OCT9_THRESHOLD_FILE,
    index=False
)

remaining_fn_detail.to_csv(
    OCT9_FN_FILE,
    index=False
)

stability_df.to_csv(
    OCT9_STABILITY_FILE,
    index=False
)

detector_expanded.to_csv(
    EXPANDED_DETECTOR_FILE,
    index=False
)

print("Saved:")
print(OCT9_CV_FILE)
print(OCT9_THRESHOLD_FILE)
print(OCT9_FN_FILE)
print(OCT9_STABILITY_FILE)
print(EXPANDED_DETECTOR_FILE)

Saved:
C:\AIML Y2 S2\Capstone 1\RYA\O1_06_expanded_cross_validation_summary.csv
C:\AIML Y2 S2\Capstone 1\RYA\O1_06_expanded_threshold_analysis.csv
C:\AIML Y2 S2\Capstone 1\RYA\O1_06_remaining_false_negative.csv
C:\AIML Y2 S2\Capstone 1\RYA\O1_06_remaining_false_negative_stability.csv
C:\AIML Y2 S2\Capstone 1\RYA\O1_06_detector_training_high_conf_477.csv


In [137]:
validation_results.to_csv(
    BASE_DIR / "O1_06_gamma_validation_500_zones_v1.csv",
    index=False
)

new_group_summary.to_csv(
    BASE_DIR / "O1_06_gamma_fit_summary_by_stratum.csv",
    index=False
)

zone_by_stratum.to_csv(
    BASE_DIR / "O1_06_compatibility_zone_by_stratum_counts.csv"
)

zone_by_stratum_pct.to_csv(
    BASE_DIR / "O1_06_compatibility_zone_by_stratum_percent.csv"
)

print("Additional validation outputs saved.")

Additional validation outputs saved.


## 19. Oct 9 milestone conclusion

The Oct 9 milestone objective was to improve the pre-inference compatibility
detector for very high recall and to characterise the cases that remain
difficult.

The larger validation experiment confirmed the main Gamma-fit failure patterns
observed in the Sep 25 pilot. Degenerate, sparse and strongly imbalanced
response scenarios showed substantially higher incompatibility risk, while
regular cases and non-diagonal-P-only cases generally remained compatible.

External validation showed that the original Sep 25 detector generalised
reasonably well but achieved only 86.2% incompatible recall on the new
high-confidence cases, demonstrating that the original pilot result was
optimistic.

After expanding the labelled development dataset to 477 high-confidence cases,
Logistic Regression achieved the highest mean incompatible recall among the
tested baseline models.

Using repeated out-of-fold predictions, a provisional threshold of 0.10
achieved 99.3% incompatible recall, detecting 143 of 144 incompatible cases.

The single remaining false negative was re-simulated across ten independent
random seeds and remained consistently above the provisional q95
incompatibility boundary. It therefore represents a genuine remaining detector
limitation.

The current detector is ready to move into the Oct 16 integration stage, where
the pre-inference screening decision can be connected to the existing neural
network and fallback pathway.